In [1]:
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    REPO_ROOT = "/content/drive/MyDrive/VidEmbedd/phase6_repo"
    import os
    if not os.path.isdir(REPO_ROOT):
        raise FileNotFoundError(
            f"{REPO_ROOT} yok - COLAB_RUNBOOK.md'ye gore ZIP'i once bu klasore cikarin.")
    os.chdir(REPO_ROOT)
    sys.path.insert(0, REPO_ROOT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
                   check=False)
    print(f"[Colab bootstrap] repo kok: {REPO_ROOT} - calisma dizini ayarlandi, "
         "bagimliliklar kuruldu.")
except ImportError:
    print("[Colab bootstrap] google.colab yok - Colab-disi ortam, ATLANDI "
         "(mevcut calisma dizini repo koku varsayiliyor).")


[Colab bootstrap] google.colab yok - Colab-disi ortam, ATLANDI (mevcut calisma dizini repo koku varsayiliyor).


# 03 - PostgreSQL metadata + telemetri (CPU/high-RAM asamasi)

Spec SS4.4. Notebook 02'nin GPU kapisindan ETKILENMEZ - bu adim yalniz
notebook 01'in GERCEK AU-AIR segment/telemetri ciktisini yukler, embedding
gerektirmez. Veritabani calisma dizini YEREL/ephemeral'dir (Drive'a DEGIL -
Colab handoff madde 5); sonuc CSV/rapor Drive'a yazilir.

**Colab'da elle bir sey yapmaniza GEREK YOK** - asagidaki hucre
`scripts/install_pgvector_colab.sh install` + `start`'i KENDISI calistirir
(apt-get + pgvector derleme, ~2-4 dk). **Yerel test/gelistirme (bu
depo):** gecici docker container `research_postgres_faz6` (port 5433) -
ana `docker-compose.yml` DEGISTIRILMEDI.

In [2]:
import json
import os
import pathlib
import subprocess
import sys

import pandas as pd
import psycopg

sys.path.insert(0, str(pathlib.Path.cwd()))
from src.research import colab_paths
from src.research.config import DEFAULT as cfg
from src.research.manifest import RunManifest, detect_hardware_profile, write_manifest

OUT = colab_paths.research_root()
hw = detect_hardware_profile()

if colab_paths.in_colab() and "PHASE6_PG_DSN" not in os.environ:
    print("Colab: PostgreSQL+pgvector kuruluyor/baslatiliyor (scripts/install_pgvector_colab.sh)...")
    for step in ("install", "start"):
        r = subprocess.run(["bash", "scripts/install_pgvector_colab.sh", step],
                           capture_output=True, text=True)
        print(r.stdout, r.stderr)
        if r.returncode != 0:
            raise RuntimeError(f"install_pgvector_colab.sh {step} basarisiz (cikti yukarida) - "
                               "environment_unavailable, bkz. scripts/colab_preflight.py raporu.")
    # script varsayilanlari: PGVECTOR_PORT=5432, PGVECTOR_DB_NAME=phase6_vector_bench,
    # --auth=trust (bu tek-kullanicili gecici VM icin sifre gerektirmez).
    os.environ["PHASE6_PG_DSN"] = "host=127.0.0.1 port=5432 dbname=phase6_vector_bench user=postgres"

# Yerel test/gelistirme icin varsayilan, mevcut docker container'a isaret eder.
DSN = os.environ.get("PHASE6_PG_DSN",
                     "host=localhost port=5433 dbname=research user=postgres password=research")
conn = psycopg.connect(DSN, autocommit=True)
print("Postgres baglantisi kuruldu:", conn.info.server_version)


Postgres baglantisi kuruldu: 160014


## Sema (spec SS4.4 - minimal sema, genisletilmedi)

In [3]:
SCHEMA_SQL = '''
DROP TABLE IF EXISTS segment_telemetry, segment_metadata, segments, videos, datasets CASCADE;

CREATE TABLE datasets (
    dataset_id      text PRIMARY KEY,
    dataset_version text NOT NULL,
    source_hash     text NOT NULL
);

CREATE TABLE videos (
    dataset_id text NOT NULL,
    video_id   text NOT NULL,
    source_uri text,
    split      text,
    duration_s double precision,
    PRIMARY KEY (dataset_id, video_id)
);

CREATE TABLE segments (
    segment_id text PRIMARY KEY,
    dataset_id text NOT NULL,
    video_id   text NOT NULL,
    t_start    double precision NOT NULL,
    t_end      double precision NOT NULL
);

CREATE TABLE segment_metadata (
    segment_id    text PRIMARY KEY REFERENCES segments(segment_id),
    person_count  integer,
    vehicle_count integer,
    bus_count     integer,
    object_classes text[],
    brightness    real,
    camera_motion real
);

CREATE TABLE segment_telemetry (
    segment_id      text PRIMARY KEY REFERENCES segments(segment_id),
    timestamp_start timestamptz,
    timestamp_end   timestamptz,
    latitude        double precision,
    longitude       double precision,
    altitude_m      real,
    velocity_mps    real,
    roll            real,
    pitch           real,
    yaw             real,
    yaw_rate        real,
    imu_summary     jsonb
);

CREATE INDEX ON segment_metadata (person_count);
CREATE INDEX ON segment_metadata (vehicle_count);
CREATE INDEX ON segment_telemetry (altitude_m);
CREATE INDEX ON segment_telemetry (velocity_mps);
'''

with conn.cursor() as cur:
    cur.execute(SCHEMA_SQL)
print("sema kuruldu (5 tablo + 4 index).")


sema kuruldu (5 tablo + 4 index).


## Yukleme (COPY) - notebook 01'in GERCEK AU-AIR ciktisi

In [4]:
seg_df = pd.read_parquet(OUT / "auair_segments.parquet")
tel_df = pd.read_parquet(OUT / "auair_telemetry.parquet")
manifest_01 = json.loads((OUT / "01_auair_download_and_validation_manifest.json").read_text(encoding="utf-8"))

with conn.cursor() as cur:
    cur.execute("INSERT INTO datasets VALUES (%s, %s, %s)",
               ("auair", "1.0-2019", manifest_01["extra"]["annotations_sha256"]))

    video_ids = seg_df["video_id"].unique()
    for vid in video_ids:
        cur.execute("INSERT INTO videos (dataset_id, video_id, split) VALUES (%s,%s,%s)",
                   ("auair", vid, "n/a"))

    for _, row in seg_df.iterrows():
        cur.execute("INSERT INTO segments VALUES (%s,%s,%s,%s,%s)",
                   (row["segment_id"], "auair", row["video_id"], row["t_start"], row["t_end"]))

    tel_by_seg = tel_df.set_index("segment_id")
    for seg_id, t in tel_by_seg.iterrows():
        cur.execute("""INSERT INTO segment_metadata (segment_id, person_count, vehicle_count)
                      VALUES (%s,%s,%s)""",
                   (seg_id, int(t["person_count"]), int(t["vehicle_count"])))
        cur.execute("""INSERT INTO segment_telemetry
                      (segment_id, altitude_m, velocity_mps, roll, pitch, yaw, yaw_rate)
                      VALUES (%s,%s,%s,%s,%s,%s,%s)""",
                   (seg_id, float(t["altitude_m"]), float(t["velocity_mps"]),
                    float(t["roll"]), float(t["pitch"]), float(t["yaw"]), float(t["yaw_rate"])))

with conn.cursor() as cur:
    counts = {}
    for table in ("datasets", "videos", "segments", "segment_metadata", "segment_telemetry"):
        cur.execute(f"SELECT count(*) FROM {table}")
        counts[table] = cur.fetchone()[0]
print(json.dumps(counts, indent=2))
assert counts["segments"] == len(seg_df), "satir sayisi dogrulamasi basarisiz"
assert counts["segment_telemetry"] == len(tel_df), "satir sayisi dogrulamasi basarisiz"
print("satir sayisi dogrulamasi GECTI (kaynak parquet == Postgres tablo satiri).")


{
  "datasets": 1,
  "videos": 8,
  "segments": 1866,
  "segment_metadata": 1866,
  "segment_telemetry": 1866
}
satir sayisi dogrulamasi GECTI (kaynak parquet == Postgres tablo satiri).


## Secicilik dogrulamasi (spec SS4.4 - her hedef seviyede GERCEKTE kac segment donuyor)

In [5]:
selectivity_thresholds = json.loads((OUT / "selectivity_thresholds.json").read_text(encoding="utf-8"))

verification_rows = []
with conn.cursor() as cur:
    for field, direction, column in [
        ("altitude_m", "less_than", "altitude_m"),
        ("velocity_mps", "greater_than", "velocity_mps"),
        ("person_count", "greater_than", "person_count"),
        ("vehicle_count", "greater_than", "vehicle_count"),
    ]:
        levels = selectivity_thresholds[field]
        table = "segment_telemetry" if column in ("altitude_m", "velocity_mps") else "segment_metadata"
        op = "<" if direction == "less_than" else ">"
        for p, info in levels.items():
            theta = info["threshold"]
            if theta is None:
                continue
            cur.execute(f"SELECT count(*) FROM {table} WHERE {column} {op} %s", (theta,))
            n_returned = cur.fetchone()[0]
            verification_rows.append({
                "field": field, "target_p": p, "threshold": theta,
                "n_returned_from_postgres": n_returned,
                "actual_selectivity_from_postgres": n_returned / counts["segments"],
                "actual_selectivity_from_derive_thresholds": info["actual_selectivity"],
            })

ver_df = pd.DataFrame(verification_rows)
ver_path = OUT / "auair_selectivity_postgres_verification.csv"
ver_df.to_csv(ver_path, index=False)
print(ver_df.to_string(index=False))
print(f"\n-> {ver_path}")


        field target_p  threshold  n_returned_from_postgres  actual_selectivity_from_postgres  actual_selectivity_from_derive_thresholds
   altitude_m      0.5  20.332150                       933                          0.500000                                   0.500000
   altitude_m      0.1  11.106789                       187                          0.100214                                   0.100214
   altitude_m     0.01   5.716373                        19                          0.010182                                   0.010182
   altitude_m    0.001   4.027586                         2                          0.001072                                   0.001072
 velocity_mps      0.5   0.103584                       933                          0.500000                                   0.500000
 velocity_mps      0.1   0.371384                       187                          0.100214                                   0.100214
 velocity_mps     0.01   1.680516        

## Ozet

In [6]:
pg_report_path = OUT / "pg_load_report.md"
pg_report = f'''# PostgreSQL yukleme raporu (GERCEK calistirmadan)

Konteyner: `research_postgres_faz6` (gecici, port 5433 - ana docker-compose.yml
DEGISTIRILMEDI, ana POC'un "Postgres yok" karari BOZULMADI).

## Satir sayilari (kaynak parquet ile birebir dogrulandi)
{json.dumps(counts, indent=2, ensure_ascii=False)}

## Secicilik dogrulamasi
`derive_thresholds()` (numpy quantile, notebook 01) ile canli Postgres
sorgusunun (`WHERE column < / > esik`) dondurdugu GERCEK satir sayisi
karsilastirildi - {len(ver_df)} satir, tam liste `auair_selectivity_postgres_verification.csv`'de.
Iki kaynak arasindaki fark yalniz kayan-nokta/quantile enterpolasyon
farklarindan kaynaklanabilir (n=8 video x pencere sayisi kucuk oldugu icin
p=0.001 gibi asiri uc seviyelerde 1-2 satirlik sapma beklenir).

## Kapsam disi
Bu notebook `segment_metadata.object_classes/brightness/camera_motion`
kolonlarini DOLDURMADI - bunlar AU-AIR annotation'inin bbox/class alanindan
turetilebilir ama spec'in bu notebook icin istedigi minimal kapsam
(person_count, vehicle_count, telemetri) ile sinirli tutuldu.
'''
pg_report_path.write_text(pg_report, encoding="utf-8")
print(pg_report)

conn.close()

manifest = RunManifest(
    notebook="03_postgres_metadata_telemetry",
    hardware_profile=hw["hardware_profile"],
    dataset_id="auair",
    extra={"row_counts": counts, "postgres_container": "research_postgres_faz6:5433"},
)
manifest_path = write_manifest(manifest, OUT)
print(f"\nmanifest -> {manifest_path}")


# PostgreSQL yukleme raporu (GERCEK calistirmadan)

Konteyner: `research_postgres_faz6` (gecici, port 5433 - ana docker-compose.yml
DEGISTIRILMEDI, ana POC'un "Postgres yok" karari BOZULMADI).

## Satir sayilari (kaynak parquet ile birebir dogrulandi)
{
  "datasets": 1,
  "videos": 8,
  "segments": 1866,
  "segment_metadata": 1866,
  "segment_telemetry": 1866
}

## Secicilik dogrulamasi
`derive_thresholds()` (numpy quantile, notebook 01) ile canli Postgres
sorgusunun (`WHERE column < / > esik`) dondurdugu GERCEK satir sayisi
karsilastirildi - 16 satir, tam liste `auair_selectivity_postgres_verification.csv`'de.
Iki kaynak arasindaki fark yalniz kayan-nokta/quantile enterpolasyon
farklarindan kaynaklanabilir (n=8 video x pencere sayisi kucuk oldugu icin
p=0.001 gibi asiri uc seviyelerde 1-2 satirlik sapma beklenir).

## Kapsam disi
Bu notebook `segment_metadata.object_classes/brightness/camera_motion`
kolonlarini DOLDURMADI - bunlar AU-AIR annotation'inin bbox/class alanindan
turetilebi